# Load Experiment Data

Reads all XDF files from `experiment_data/` and converts each LSL stream into a pandas DataFrame.

## LSL Stream Channels

Each XDF file contains up to 7 LSL streams produced by the Seesaw Unity project:

| Stream | Channels | Description |
|---|---|---|
| `Unity.GameManagerState` | `is_pause` (0/1), `leading_player_id` (+1 or -1), `play_score` (int) | Game state: pause flag, which player leads the current block, cumulative score |
| `Unity.Position.Player1` | `pos_x`, `pos_y`, `pos_z` | World position of Player 1 (server-side player) |
| `Unity.Position.Player2` | `pos_x`, `pos_y`, `pos_z` | World position of Player 2 (client-side player) |
| `Unity.Position.Target1` | `pos_x`, `pos_y`, `pos_z` | World position of Target 1 (belongs to Player 1) |
| `Unity.Position.Target2` | `pos_x`, `pos_y`, `pos_z` | World position of Target 2 (belongs to Player 2) |
| `Unity.Position.Ball` | `pos_x`, `pos_y`, `pos_z` | World position of the seesaw ball |
| `Unity.ClientPlayerDistance` | `player_shift_distance` | Per-frame movement delta from client player input (BasePlayerSpeed × direction × acceleration) |

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyxdf

## Configuration

In [ ]:
DATA_DIR = Path("experiment_data")

# Column name mapping for each stream type
STREAM_COLUMNS = {
    "Unity.GameManagerState": ["is_pause", "leading_player_id", "play_score"],
    "Unity.Position": ["pos_x", "pos_y", "pos_z"],
    "Unity.ClientPlayerDistance": ["player_shift_distance"],
}

## Helper Functions

In [ ]:
def get_columns_for_stream(stream_name: str) -> list[str]:
    """Return column names for a given LSL stream name."""
    if stream_name.startswith("Unity.Position."):
        return STREAM_COLUMNS["Unity.Position"]
    return STREAM_COLUMNS.get(stream_name, [])


def parse_filename(filename: str) -> dict:
    """Extract date, dyad ID, and session number from filename.

    Expected format: 20260413_1330_Session1.xdf
    """
    stem = Path(filename).stem
    parts = stem.split("_")
    return {
        "date": parts[0],
        "dyad": parts[1],
        "session": int(parts[2].replace("Session", "")),
    }


def stream_to_dataframe(stream: dict) -> pd.DataFrame:
    """Convert a single XDF stream dict to a pandas DataFrame."""
    stream_name = stream["info"]["name"][0]
    columns = get_columns_for_stream(stream_name)

    if not columns:
        # Unknown stream — use generic column names
        n_channels = int(stream["info"]["channel_count"][0])
        columns = [f"ch_{i}" for i in range(n_channels)]

    df = pd.DataFrame(stream["time_series"], columns=columns)
    df.insert(0, "timestamp", stream["time_stamps"])
    return df


def load_xdf_to_dataframes(filepath: Path) -> dict[str, pd.DataFrame]:
    """Load an XDF file and return a dict mapping stream names to DataFrames."""
    data, _ = pyxdf.load_xdf(str(filepath))
    result = {}
    for stream in data:
        stream_name = stream["info"]["name"][0]
        result[stream_name] = stream_to_dataframe(stream)
    return result

## Load All XDF Files

The result is a nested dictionary: `all_data[dyad][session][stream_name]` → `DataFrame`

In [ ]:
xdf_files = sorted(DATA_DIR.glob("*.xdf"))
print(f"Found {len(xdf_files)} XDF files")

all_data = {}

for filepath in xdf_files:
    meta = parse_filename(filepath.name)
    dyad = meta["dyad"]
    session = meta["session"]

    streams = load_xdf_to_dataframes(filepath)

    if dyad not in all_data:
        all_data[dyad] = {}
    all_data[dyad][session] = streams

    stream_summary = ", ".join(f"{name} ({len(df)} rows)" for name, df in streams.items())
    print(f"  {filepath.name}: {stream_summary}")

print(f"\nLoaded {len(all_data)} dyads: {sorted(all_data.keys())}")

## Inspect a Sample DataFrame

Pick the first dyad's first session to show what the data looks like.

In [ ]:
first_dyad = sorted(all_data.keys())[0]
first_session = sorted(all_data[first_dyad].keys())[0]
streams = all_data[first_dyad][first_session]

print(f"Dyad: {first_dyad}, Session: {first_session}")
print(f"Streams: {list(streams.keys())}\n")

for name, df in streams.items():
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    duration = df["timestamp"].iloc[-1] - df["timestamp"].iloc[0]
    print(f"Duration: {duration:.1f}s")
    print(df.head(3))
    print()

## Build Flat Summary Table

One row per file with recording metadata for quick overview.

In [ ]:
summary_rows = []

for dyad in sorted(all_data.keys()):
    for session in sorted(all_data[dyad].keys()):
        streams = all_data[dyad][session]
        # Use the first stream's timestamps for duration
        first_stream = next(iter(streams.values()))
        duration = first_stream["timestamp"].iloc[-1] - first_stream["timestamp"].iloc[0]
        summary_rows.append({
            "dyad": dyad,
            "session": session,
            "n_streams": len(streams),
            "duration_s": round(duration, 1),
            "has_ball": "Unity.Position.Ball" in streams,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df